In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

# Project root
PROJECT_ROOT = Path.cwd().parent

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Project root:", PROJECT_ROOT)
print("PyTorch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Project root: c:\Users\sanka\OneDrive\Desktop\research paper
PyTorch: 2.14.1+cpu
Device: cpu


In [3]:
from datasets import load_dataset

DATASET_NAME = "ai4bharat/IndicMSMARCO"
LANG = "hi"

ds = load_dataset(DATASET_NAME, LANG)

print(ds)

DatasetDict({
    train: Dataset({
        features: ['query_id', 'query', 'passage', 'passage_id', 'language', 'answer', 'title', 'url', 'query_type', 'relevance_score', 'is_selected', 'text', 'meta', 'dataset', 'source'],
        num_rows: 1000
    })
})


In [4]:
print("Splits:", ds.keys())

data = ds["train"]

print("Number of rows:", len(data))
print("Columns:")
print(data.column_names)

Splits: dict_keys(['train'])
Number of rows: 1000
Columns:
['query_id', 'query', 'passage', 'passage_id', 'language', 'answer', 'title', 'url', 'query_type', 'relevance_score', 'is_selected', 'text', 'meta', 'dataset', 'source']


In [5]:
row = data[0]

for key, value in row.items():
    print(f"{key}: {value}")

query_id: 10440
query: हिरलूम टमाटर का क्या अर्थ है
passage: विरासती बागवानों को यह बात पता है, कि खुले परागण की अवधारणा थोड़ी भ्रामक है, क्योंकि कई विरासती सब्जियों के परागण में कुछ भी खुला नहीं है। उदाहरण के लिए, स्क्वैश और कद्दू लें। अब तक, इस क्षेत्र के विशेषज्ञ इस बात से सहमत हैं कि विरासती सब्जियां पुरानी, खुले परागण वाली किस्में हैं। इसके अलावा, इन किस्मों को उच्च गुणवत्ता वाला और आसानी से उगाने वाला भी माना जाता है। शायद विस्तार से चर्चा करने के लिए यह सबसे अच्छा होगा कि प्रत्येक विशेषता को अलग-अलग देखा जाए।
passage_id: 
language: hi
answer: टमाटर की एक खुली परागण (गैर-संकर) विरासत किस्म।
title: 
url: 
query_type: description
relevance_score: 1.0
is_selected: True
text: Query: हिरलूम टमाटर का क्या अर्थ है | Passage: विरासती बागवानों को यह बात पता है, कि खुले परागण की अवधारणा थोड़ी भ्रामक है, क्योंकि कई विरासती सब्जियों के परागण में कुछ भी खुला नहीं है। उदाहरण के लिए, स्क्वैश और कद्दू लें। अब तक, इस क्षेत्र के विशेषज्ञ इस बात से सहमत हैं कि विरासती सब्जियां पुरानी, खुले परागण वा

In [6]:
query  = data["query"]
passage = data["passage"]

In [8]:
# Step 2: Prepare Hindi benchmark

from datasets import load_dataset
import pandas as pd
import hashlib

DATASET_NAME = "ai4bharat/IndicMSMARCO"
LANG = "hi"

ds = load_dataset(DATASET_NAME, LANG)
data = ds["train"]

print("Rows:", len(data))
print("Columns:", data.column_names)

Rows: 1000
Columns: ['query_id', 'query', 'passage', 'passage_id', 'language', 'answer', 'title', 'url', 'query_type', 'relevance_score', 'is_selected', 'text', 'meta', 'dataset', 'source']


In [9]:
queries = data["query"]
passages = data["passage"]

print("Number of queries:", len(queries))
print("Number of passages:", len(passages))

print("\nExample query:")
print(queries[0])

print("\nExample passage:")
print(passages[0])

Number of queries: 1000
Number of passages: 1000

Example query:
हिरलूम टमाटर का क्या अर्थ है

Example passage:
विरासती बागवानों को यह बात पता है, कि खुले परागण की अवधारणा थोड़ी भ्रामक है, क्योंकि कई विरासती सब्जियों के परागण में कुछ भी खुला नहीं है। उदाहरण के लिए, स्क्वैश और कद्दू लें। अब तक, इस क्षेत्र के विशेषज्ञ इस बात से सहमत हैं कि विरासती सब्जियां पुरानी, खुले परागण वाली किस्में हैं। इसके अलावा, इन किस्मों को उच्च गुणवत्ता वाला और आसानी से उगाने वाला भी माना जाता है। शायद विस्तार से चर्चा करने के लिए यह सबसे अच्छा होगा कि प्रत्येक विशेषता को अलग-अलग देखा जाए।


In [10]:
print("query_id:", data["query_id"][:5])
print("passage_id:", data["passage_id"][:5])

print("\nFirst relevance scores:")
print(data["relevance_score"][:10])

print("\nFirst is_selected values:")
print(data["is_selected"][:10])

query_id: ['10440', '12134', '10077', '12599', '12430']
passage_id: ['', '', '', '', '']

First relevance scores:
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

First is_selected values:
[True, True, True, True, True, True, True, True, True, True]


In [11]:
# Build corpus from unique passages

corpus_df = pd.DataFrame({
    "passage_id": data["passage_id"],
    "passage": data["passage"]
})

corpus_df = corpus_df.drop_duplicates(subset="passage")

print("Original rows:", len(data))
print("Unique corpus passages:", len(corpus_df))

corpus_df.head()

Original rows: 1000
Unique corpus passages: 1000


,passage_id,passage
0,,"विरासती बागवानों को यह बात पता है, कि खुले परा..."
1,,ब्रसेल्स स्प्राउट्स को अपने रेफ्रिजरेटर के क्र...
2,,मायस्थेनिया ग्रेविस एक पुरानी ऑटोइम्यून न्यूरो...
3,,"कागज़ उद्योग में, पॉलीविनाइल अल्कोहल का उपयोग ..."
4,,शिक्षण शुल्क. आयोवा विश्वविद्यालय के शिक्षण शु...


In [12]:
# Verify there are no empty passages

empty_passages = corpus_df["passage"].isna().sum()
empty_strings = (corpus_df["passage"].str.strip() == "").sum()

print("Missing passages:", empty_passages)
print("Empty passages:", empty_strings)

Missing passages: 0
Empty passages: 0


In [13]:
print("Rows:", len(data))

print("Queries:", len(data["query"]))
print("Passages:", len(data["passage"]))

print("First query:")
print(repr(data["query"][0]))

print("\nFirst passage:")
print(repr(data["passage"][0]))

print("\nUnique queries:", len(set(data["query"])))
print("Unique passages:", len(set(data["passage"])))

print("\nMissing passages:", sum(x is None for x in data["passage"]))
print("Empty passages:", sum(not str(x).strip() for x in data["passage"]))

Rows: 1000
Queries: 1000
Passages: 1000
First query:
'हिरलूम टमाटर का क्या अर्थ है'

First passage:
'विरासती बागवानों को यह बात पता है, कि खुले परागण की अवधारणा थोड़ी भ्रामक है, क्योंकि कई विरासती सब्जियों के परागण में कुछ भी खुला नहीं है। उदाहरण के लिए, स्क्वैश और कद्दू लें। अब तक, इस क्षेत्र के विशेषज्ञ इस बात से सहमत हैं कि विरासती सब्जियां पुरानी, खुले परागण वाली किस्में हैं। इसके अलावा, इन किस्मों को उच्च गुणवत्ता वाला और आसानी से उगाने वाला भी माना जाता है। शायद विस्तार से चर्चा करने के लिए यह सबसे अच्छा होगा कि प्रत्येक विशेषता को अलग-अलग देखा जाए।'

Unique queries: 1000
Unique passages: 1000

Missing passages: 0
Empty passages: 0


In [14]:
query_ids = data["query_id"]
passage_ids = data["passage_id"]

qrels = {}

for qid, pid in zip(query_ids, passage_ids):
    qrels[qid] = pid

print("Number of queries in qrels:", len(qrels))

for qid, pid in list(qrels.items())[:5]:
    print(f"{qid} -> {pid}")

Number of queries in qrels: 1000
10440 -> 
12134 -> 
10077 -> 
12599 -> 
12430 -> 


In [15]:
# Check that query_id and passage_id correspond row-by-row

for i in range(5):
    print(f"\nQuery ID: {data['query_id'][i]}")
    print(f"Query: {data['query'][i]}")
    print(f"Gold passage ID: {data['passage_id'][i]}")
    print(f"Passage: {data['passage'][i][:200]}...")


Query ID: 10440
Query: हिरलूम टमाटर का क्या अर्थ है
Gold passage ID: 
Passage: विरासती बागवानों को यह बात पता है, कि खुले परागण की अवधारणा थोड़ी भ्रामक है, क्योंकि कई विरासती सब्जियों के परागण में कुछ भी खुला नहीं है। उदाहरण के लिए, स्क्वैश और कद्दू लें। अब तक, इस क्षेत्र के विश...

Query ID: 12134
Query: ब्रसेल्स स्प्राउट्स को फ्रिज में कितने दिन तक रखा जा सकता है
Gold passage ID: 
Passage: ब्रसेल्स स्प्राउट्स को अपने रेफ्रिजरेटर के क्रिस्पर में एक प्लास्टिक बैग में स्टोर करें, जहां वे कम से कम एक सप्ताह तक ताज़ा रहते हैं, अगर थोड़ा अधिक नहीं। तने पर अभी भी लगे हुए स्प्राउट्स व्यक्तिगत र...

Query ID: 10077
Query: मायस्थेनिया ग्रेविस का इलाज क्या है?
Gold passage ID: 
Passage: मायस्थेनिया ग्रेविस एक पुरानी ऑटोइम्यून न्यूरोमस्कुलर बीमारी है, जिसकी विशेषता शरीर की कंकाल (स्वैच्छिक) मांसपेशियों की कमजोरी की विभिन्न डिग्री है। मायस्थेनिया ग्रेविस मांसपेशियों तक तंत्रिका आवेगों ...

Query ID: 12599
Query: पीवीए का उपयोग किस लिए किया जाता है?
Gold passage ID: 
Passage: कागज़ उद्योग में, पॉ

In [16]:
import hashlib
import re

def normalize_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

def make_doc_id(text):
    normalized = normalize_text(text)
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()

In [17]:
# Build corpus

corpus = {}

for passage in data["passage"]:
    passage_clean = normalize_text(passage)
    doc_id = make_doc_id(passage_clean)

    corpus[doc_id] = passage_clean

print("Corpus documents:", len(corpus))

Corpus documents: 1000


In [18]:
# Map each query to the hashed document ID

qrels_doc = {}

for qid, passage in zip(data["query_id"], data["passage"]):
    doc_id = make_doc_id(passage)
    qrels_doc[qid] = doc_id

print("Queries:", len(qrels_doc))

Queries: 1000


In [19]:
# Critical benchmark invariant

missing_gold = [
    qid for qid, doc_id in qrels_doc.items()
    if doc_id not in corpus
]

print("Missing gold documents:", len(missing_gold))

assert len(missing_gold) == 0

print("✓ Every query has a gold document in the corpus")

Missing gold documents: 0
✓ Every query has a gold document in the corpus


In [20]:
import random

SEED = 42

corpus_items = list(corpus.items())

random.seed(SEED)
random.shuffle(corpus_items)

corpus = dict(corpus_items)

print("Corpus size after shuffle:", len(corpus))

Corpus size after shuffle: 1000


In [21]:
assert len(corpus) == 1000
assert len(qrels_doc) == 1000

print("✓ Corpus invariant:", len(corpus))
print("✓ Qrels invariant:", len(qrels_doc))

✓ Corpus invariant: 1000
✓ Qrels invariant: 1000


In [22]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-m3"

model = SentenceTransformer(
    MODEL_NAME,
    device="cpu"
)

model.max_seq_length = 512

print("Model:", MODEL_NAME)
print("Max sequence length:", model.max_seq_length)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model: BAAI/bge-m3
Max sequence length: 512


In [23]:
tokenizer = model.tokenizer

for lang in ["hi", "ta", "as"]:
    sample_text = {
        "hi": "यह एक हिंदी परीक्षण वाक्य है।",
        "ta": "இது ஒரு தமிழ் சோதனை வாக்கியம்.",
        "as": "এইটো এটা অসমীয়া পৰীক্ষামূলক বাক্য।"
    }[lang]

    tokens = tokenizer(
        sample_text,
        truncation=True,
        max_length=512
    )

    print(f"{lang}: {len(tokens['input_ids'])} tokens")

hi: 9 tokens
ta: 9 tokens
as: 14 tokens


In [24]:
import time

pilot_passages = list(corpus.values())[:64]

start = time.time()

pilot_embeddings = model.encode(
    pilot_passages,
    batch_size=8,
    normalize_embeddings=True,
    show_progress_bar=True
)

elapsed = time.time() - start

print("Embedding shape:", pilot_embeddings.shape)
print(f"Time for 64 passages: {elapsed:.2f} seconds")

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Embedding shape: (64, 1024)
Time for 64 passages: 71.80 seconds


In [25]:
# Convert corpus dictionary to ordered lists

doc_ids = list(corpus.keys())
documents = list(corpus.values())

print("Documents:", len(documents))
print("Doc IDs:", len(doc_ids))

Documents: 1000
Doc IDs: 1000


In [26]:
import time

start = time.time()

corpus_embeddings = model.encode(
    documents,
    batch_size=8,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)

elapsed = time.time() - start

print("Corpus embeddings shape:", corpus_embeddings.shape)
print(f"Encoding time: {elapsed / 60:.2f} minutes")

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Corpus embeddings shape: (1000, 1024)
Encoding time: 8.31 minutes


In [27]:
import numpy as np

norms = np.linalg.norm(corpus_embeddings, axis=1)

print("Minimum norm:", norms.min())
print("Maximum norm:", norms.max())

assert np.allclose(norms, 1.0, atol=1e-5)

print("✓ Corpus embeddings are L2 normalized")

Minimum norm: 0.9999999
Maximum norm: 1.0000001
✓ Corpus embeddings are L2 normalized


In [28]:
query_ids = data["query_id"]
queries = data["query"]

start = time.time()

query_embeddings = model.encode(
    queries,
    batch_size=8,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)

elapsed = time.time() - start

print("Query embeddings shape:", query_embeddings.shape)
print(f"Query encoding time: {elapsed / 60:.2f} minutes")

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Query embeddings shape: (1000, 1024)
Query encoding time: 1.12 minutes


In [29]:
query_norms = np.linalg.norm(query_embeddings, axis=1)

print("Minimum norm:", query_norms.min())
print("Maximum norm:", query_norms.max())

assert np.allclose(query_norms, 1.0, atol=1e-5)

print("✓ Query embeddings are L2 normalized")

Minimum norm: 0.9999998
Maximum norm: 1.0000001
✓ Query embeddings are L2 normalized


In [30]:
# Compute query-document cosine similarities

similarities = query_embeddings @ corpus_embeddings.T

print("Similarity matrix shape:", similarities.shape)

Similarity matrix shape: (1000, 1000)


In [31]:
# Rank documents for each query

ranked_indices = np.argsort(-similarities, axis=1)

print("Ranking matrix shape:", ranked_indices.shape)

Ranking matrix shape: (1000, 1000)


In [32]:
ranked_doc_ids = [
    [doc_ids[idx] for idx in row]
    for row in ranked_indices
]

print("Number of ranked queries:", len(ranked_doc_ids))
print("Documents ranked for first query:", len(ranked_doc_ids[0]))

Number of ranked queries: 1000
Documents ranked for first query: 1000


In [33]:
# Find the rank of the correct document

gold_ranks = []

for i, qid in enumerate(query_ids):
    gold_doc = qrels_doc[qid]

    ranked_docs = ranked_doc_ids[i]

    rank = ranked_docs.index(gold_doc) + 1
    gold_ranks.append(rank)

gold_ranks = np.array(gold_ranks)

print("Minimum rank:", gold_ranks.min())
print("Maximum rank:", gold_ranks.max())
print("Mean rank:", gold_ranks.mean())

Minimum rank: 1
Maximum rank: 771
Mean rank: 5.738


In [34]:
# Mean Reciprocal Rank

reciprocal_ranks = 1.0 / gold_ranks

mrr = reciprocal_ranks.mean()

print(f"MRR: {mrr:.4f}")

MRR: 0.8863


In [35]:
# Recall@K

for k in [1, 5, 10, 20, 100]:
    recall = np.mean(gold_ranks <= k)
    print(f"Recall@{k}: {recall:.4f}")

Recall@1: 0.8400
Recall@5: 0.9440
Recall@10: 0.9570
Recall@20: 0.9690
Recall@100: 0.9910


In [36]:
# Save Hindi BGE-M3 baseline result

result = {
    "model": "BAAI/bge-m3",
    "language": "hi",
    "corpus_size": len(corpus),
    "num_queries": len(queries),
    "mrr": float(mrr),
    "recall@1": float(np.mean(gold_ranks <= 1)),
    "recall@5": float(np.mean(gold_ranks <= 5)),
    "recall@10": float(np.mean(gold_ranks <= 10)),
    "recall@20": float(np.mean(gold_ranks <= 20)),
    "recall@100": float(np.mean(gold_ranks <= 100)),
}

print(result)

{'model': 'BAAI/bge-m3', 'language': 'hi', 'corpus_size': 1000, 'num_queries': 1000, 'mrr': 0.8862929424512407, 'recall@1': 0.84, 'recall@5': 0.944, 'recall@10': 0.957, 'recall@20': 0.969, 'recall@100': 0.991}


In [37]:
import json
from pathlib import Path

Path("../results").mkdir(exist_ok=True)

with open("../results/bge_m3_hi.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2)

print("Saved: ../results/bge_m3_hi.json")

Saved: ../results/bge_m3_hi.json


In [38]:
# ============================================================
# BGE-M3: Full IndicMSMARCO multilingual baseline
# ============================================================

import gc
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset

DATASET_NAME = "ai4bharat/IndicMSMARCO"
MODEL_NAME = "BAAI/bge-m3"

LANGUAGES = [
    "as",  # Assamese
    "bn",  # Bengali
    "gu",  # Gujarati
    "hi",  # Hindi
    "kn",  # Kannada
    "ml",  # Malayalam
    "mr",  # Marathi
    "ne",  # Nepali
    "or",  # Odia
    "pa",  # Punjabi
    "ta",  # Tamil
    "te",  # Telugu
    "ur",  # Urdu
]

SEED = 42
BATCH_SIZE = 8

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)


model.max_seq_length = 512

print("=" * 70)
print("BGE-M3 MULTILINGUAL BASELINE")
print("=" * 70)
print(f"Model: {MODEL_NAME}")
print(f"Languages: {len(LANGUAGES)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max sequence length: {model.max_seq_length}")
print("=" * 70)


def evaluate_language(lang):

    print("\n" + "=" * 70)
    print(f"Evaluating: {lang}")
    print("=" * 70)

    total_start = time.time()


    ds = load_dataset(DATASET_NAME, lang)
    data = ds["train"]

    print(f"Rows: {len(data)}")


    queries = data["query"]
    passages = data["passage"]
    query_ids = data["query_id"]

    assert len(queries) == len(passages)
    assert len(queries) == len(query_ids)

    assert len(set(query_ids)) == len(query_ids), \
        "Duplicate query IDs detected"

    assert all(
        p is not None and str(p).strip()
        for p in passages
    ), "Missing/empty passage detected"

    print(f"Queries: {len(queries)}")
    print(f"Passages: {len(passages)}")
    print(f"Unique queries: {len(set(queries))}")
    print(f"Unique passages: {len(set(passages))}")


    corpus = {}

    for passage in passages:

        passage_clean = normalize_text(passage)
        doc_id = make_doc_id(passage_clean)

        corpus[doc_id] = passage_clean

    print(f"Corpus documents: {len(corpus)}")

    # Benchmark invariant
    assert len(corpus) == len(passages), \
        f"Expected {len(passages)} unique passages, got {len(corpus)}"


    qrels = {}

    for qid, passage in zip(query_ids, passages):

        passage_clean = normalize_text(passage)
        doc_id = make_doc_id(passage_clean)

        qrels[qid] = doc_id

    assert len(qrels) == len(query_ids)

    # Verify every gold document exists
    missing_gold = [
        qid
        for qid, doc_id in qrels.items()
        if doc_id not in corpus
    ]

    assert len(missing_gold) == 0, \
        f"{len(missing_gold)} gold documents missing"

    print("✓ Corpus/qrels validation passed")

    # --------------------------------------------------------
    # 5. Shuffle corpus order
    # --------------------------------------------------------

    corpus_items = list(corpus.items())

    random.seed(SEED)
    random.shuffle(corpus_items)

    corpus = dict(corpus_items)

    doc_ids = list(corpus.keys())
    documents = list(corpus.values())

    # --------------------------------------------------------
    # 6. Encode corpus
    # --------------------------------------------------------

    start = time.time()

    corpus_embeddings = model.encode(
        documents,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    corpus_time = time.time() - start

    print(
        f"Corpus embeddings: {corpus_embeddings.shape}"
    )
    print(
        f"Corpus encoding time: {corpus_time / 60:.2f} min"
    )

    # Verify dimensions
    assert corpus_embeddings.shape[0] == len(documents)
    assert corpus_embeddings.shape[1] == 1024

    # Verify normalization
    corpus_norms = np.linalg.norm(
        corpus_embeddings,
        axis=1
    )

    assert np.allclose(
        corpus_norms,
        1.0,
        atol=1e-5
    )

    # --------------------------------------------------------
    # 7. Encode queries
    # --------------------------------------------------------

    start = time.time()

    query_embeddings = model.encode(
        queries,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    query_time = time.time() - start

    print(
        f"Query embeddings: {query_embeddings.shape}"
    )
    print(
        f"Query encoding time: {query_time / 60:.2f} min"
    )

    assert query_embeddings.shape[0] == len(queries)
    assert query_embeddings.shape[1] == 1024

    # Verify normalization
    query_norms = np.linalg.norm(
        query_embeddings,
        axis=1
    )

    assert np.allclose(
        query_norms,
        1.0,
        atol=1e-5
    )


    start = time.time()

    # Normalized embeddings -> dot product = cosine similarity
    similarities = query_embeddings @ corpus_embeddings.T

    retrieval_time = time.time() - start

    print(
        f"Similarity matrix: {similarities.shape}"
    )
    print(
        f"Retrieval time: {retrieval_time:.2f} sec"
    )


    ranked_indices = np.argsort(
        -similarities,
        axis=1
    )


    gold_ranks = []

    for i, qid in enumerate(query_ids):

        gold_doc = qrels[qid]

        ranked_indices_for_query = ranked_indices[i]

        # Find the position of the gold document
        gold_index = doc_ids.index(gold_doc)

        positions = np.where(
            ranked_indices_for_query == gold_index
        )[0]

        assert len(positions) == 1

        rank = int(positions[0]) + 1

        gold_ranks.append(rank)

    gold_ranks = np.array(
        gold_ranks,
        dtype=np.int32
    )


    mrr = np.mean(1.0 / gold_ranks)

    recall_1 = np.mean(gold_ranks <= 1)
    recall_5 = np.mean(gold_ranks <= 5)
    recall_10 = np.mean(gold_ranks <= 10)
    recall_20 = np.mean(gold_ranks <= 20)
    recall_100 = np.mean(gold_ranks <= 100)

    total_time = time.time() - total_start


    result = {
        "model": MODEL_NAME,
        "language": lang,
        "num_queries": len(queries),
        "corpus_size": len(corpus),
        "mrr": float(mrr),
        "recall@1": float(recall_1),
        "recall@5": float(recall_5),
        "recall@10": float(recall_10),
        "recall@20": float(recall_20),
        "recall@100": float(recall_100),
        "corpus_encoding_minutes": float(corpus_time / 60),
        "query_encoding_minutes": float(query_time / 60),
        "retrieval_seconds": float(retrieval_time),
        "total_minutes": float(total_time / 60),
    }


    output_file = RESULTS_DIR / f"bge_m3_{lang}.json"

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            result,
            f,
            indent=2,
            ensure_ascii=False
        )


    print("\n" + "-" * 70)
    print(f"RESULT: {lang}")
    print("-" * 70)
    print(f"MRR:       {mrr:.4f}")
    print(f"Recall@1:  {recall_1:.4f}")
    print(f"Recall@5:  {recall_5:.4f}")
    print(f"Recall@10: {recall_10:.4f}")
    print(f"Recall@20: {recall_20:.4f}")
    print(f"Recall@100:{recall_100:.4f}")
    print(f"Saved: {output_file}")
    print("-" * 70)


    del ds
    del data
    del corpus
    del corpus_embeddings
    del query_embeddings
    del similarities
    del ranked_indices

    gc.collect()

    return result


all_results = []

for lang in LANGUAGES:

    try:

        result = evaluate_language(lang)
        all_results.append(result)

    except Exception as e:

        print("\n" + "!" * 70)
        print(f"FAILED: {lang}")
        print(f"ERROR: {repr(e)}")
        print("!" * 70)

        continue


results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    "language"
).reset_index(drop=True)

results_df.to_csv(
    RESULTS_DIR / "bge_m3_all_languages.csv",
    index=False
)

print("\n" + "=" * 70)
print("FINAL BGE-M3 RESULTS")
print("=" * 70)

print(
    results_df[
        [
            "language",
            "mrr",
            "recall@1",
            "recall@5",
            "recall@10",
            "recall@20",
            "recall@100",
        ]
    ].to_string(index=False)
)

print("\nSaved:")
print(RESULTS_DIR / "bge_m3_all_languages.csv")

BGE-M3 MULTILINGUAL BASELINE
Model: BAAI/bge-m3
Languages: 13
Batch size: 8
Max sequence length: 512

Evaluating: as
Rows: 1000
Queries: 1000
Passages: 1000
Unique queries: 1000
Unique passages: 1000
Corpus documents: 1000
✓ Corpus/qrels validation passed


Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Corpus embeddings: (1000, 1024)
Corpus encoding time: 7.75 min


Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Query embeddings: (1000, 1024)
Query encoding time: 0.97 min
Similarity matrix: (1000, 1000)
Retrieval time: 0.02 sec

----------------------------------------------------------------------
RESULT: as
----------------------------------------------------------------------
MRR:       0.8073
Recall@1:  0.7460
Recall@5:  0.8750
Recall@10: 0.9110
Recall@20: 0.9360
Recall@100:0.9770
Saved: ..\results\bge_m3_as.json
----------------------------------------------------------------------

Evaluating: bn
Rows: 1000
Queries: 1000
Passages: 1000
Unique queries: 1000
Unique passages: 1000
Corpus documents: 1000
✓ Corpus/qrels validation passed


...

Query embeddings: (1000, 1024)
Query encoding time: 0.92 min
Similarity matrix: (1000, 1000)
Retrieval time: 0.01 sec

----------------------------------------------------------------------
RESULT: te
----------------------------------------------------------------------
MRR:       0.8611
Recall@1:  0.8120
Recall@5:  0.9190
Recall@10: 0.9420
Recall@20: 0.9580
Recall@100:0.9830
Saved: ..\results\bge_m3_te.json
----------------------------------------------------------------------

Evaluating: ur
Rows: 1000
Queries: 1000
Passages: 1000
Unique queries: 1000
Unique passages: 1000
Corpus documents: 1000
✓ Corpus/qrels validation passed


Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Corpus embeddings: (1000, 1024)
Corpus encoding time: 8.09 min


Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Query embeddings: (1000, 1024)
Query encoding time: 1.99 min
Similarity matrix: (1000, 1000)
Retrieval time: 0.02 sec

----------------------------------------------------------------------
RESULT: ur
----------------------------------------------------------------------
MRR:       0.8365
Recall@1:  0.7810
Recall@5:  0.9010
Recall@10: 0.9200
Recall@20: 0.9490
Recall@100:0.9790
Saved: ..\results\bge_m3_ur.json
----------------------------------------------------------------------

FINAL BGE-M3 RESULTS
language      mrr  recall@1  recall@5  recall@10  recall@20  recall@100
      as 0.807262  0.746000  0.875000   0.911000   0.936000    0.977000
      bn 0.860217  0.811000  0.913000   0.943000   0.964000    0.986000
      gu 0.833590  0.767768  0.913914   0.936937   0.952953    0.978979
      hi 0.886293  0.840000  0.944000   0.957000   0.969000    0.991000
      kn 0.837893  0.786000  0.900000   0.920000   0.937000    0.979000
      ml 0.840472  0.788000  0.900000   0.927000   0.946000   